# Verified Voltage Control for AI-Training Data Centers

This notebook installs the dependencies and runs the core verification end to end on a free CPU Colab: the **exact** certificate for the paper's switching-reference controller (Theorem 1), and the **CROWN** certificate for a neural controller trained in the same setting (sound for every input in an operating box).

Paper: Yan, Joswig-Jones, Zhang, Chen, Cui, *Switching-Reference Voltage Control for Distribution Systems with AI-Training Data Centers*, arXiv:2603.15588.

In [ ]:
# 1. Dependencies (CPU-only). auto_LiRPA's metadata pins Python 3.11 but the code runs elsewhere.
!pip -q install "torch==2.5.1" --index-url https://download.pytorch.org/whl/cpu
!pip -q install numpy scipy matplotlib pyyaml
!pip -q install --ignore-requires-python "git+https://github.com/Verified-Intelligence/auto_LiRPA.git"

In [ ]:
# 2. Get the code.
![ -d verified-voltage-control ] || git clone https://github.com/sehajr-singhs/verified-voltage-control.git
%cd verified-voltage-control

In [ ]:
# 3. Self-tests: grid model, controllers, Theorem 1 gain condition.
!python -m pytest -q tests

## Exact certificate (E3): the published controller is affine, so its safety is closed form

In [ ]:
import numpy as np
from src.grid import Feeder
from src.load import DCWorkload
from src.control import max_stable_gain
from src.certify_exact import certified_envelope

f = Feeder()
k = 0.7 * max_stable_gain(f.X)
wl = DCWorkload([22], compute_pu=0.40, comm_pu=0.03, compute_steps=60, comm_steps=20, wbar=0.006)
cert = certified_envelope(f, wl, k)
print('gain condition 0 < K < 2X^-1 satisfied:', cert['gain_condition']['satisfied'])
print('rho(I - XK) = %.5f' % cert['contraction']['rho'])
print('certified per-bus envelope max = %.4f (sound worst case)' % cert['env_max'])

## CROWN certificate (E4): the neural controller, proven for EVERY input in the box

In [ ]:
# Train the neural policy (behavior cloning + safety penalty), then certify one step with CROWN.
from src.config import load_config
from src.neural import collect_clone_data, train_policy_safe, operating_box
from src.certify_crown import make_bounded, certify, empirical_sweep, band_certified, tightness

cfg = load_config(); dc = [22]; n = len(f.buses)
E, Q = collect_clone_data(cfg, f, dc, seeds=[0, 1, 2])
e_c, e_r_tr, p_c, p_r = operating_box(cfg, f, dc, cfg['neural']['train_radius'])
policy, onestep, mse = train_policy_safe(f, E, Q, dc, p_c, p_r, e_r_tr, epochs=600)
print('behavior-cloning MSE = %.2e' % mse)

bm = make_bounded(onestep, e_c, p_c)
r = 0.04
er = np.full(n, r)
cert = certify(bm, e_c, er, p_c, p_r)
emp_lb, emp_ub = empirical_sweep(onestep, e_c, er, p_c, p_r, 50000)
for m, (lb, ub) in cert.items():
    print('%-12s band certified over the WHOLE box: %-5s  (bound is %.2fx the empirical worst case)'
          % (m, band_certified(lb, ub), tightness(lb, ub, emp_lb, emp_ub)))

IBP cannot certify the band (its bound is off-scale), CROWN and alpha-CROWN both certify that for **all** voltage states within 0.04 p.u. and **all** data-center loads across the mode range, one step keeps every bus inside +/-5%. This is a guarantee for every input in the box, not a sampled fraction.

Run the complete pipeline (all five experiments, figures, master table) with `!python -m experiments.run_all --quick`.